In [1]:
import prun_data_frames as prundf
from prun_data_frames import CX, Currency, PrunCXPCAll
from price import EvaluatedPrices, PriceOverride, PriceType, OverrideType
from labor import EvaluatedLabor
from analysis import BuildingAnalysis
from config import Config
import polars as pl
import polars.selectors as cs
import os

In [2]:
prices = prundf.PrunPrices()

overrides = []

# enter price overrides here
# overrides = [PriceOverride("AR",CX.CI1,20,OverrideType.BUY)]

cx = CX.CI1

this_file = f"{os.getcwd()}/analysis.ipynb"

cfg = Config(this_file)

cxpc_all = PrunCXPCAll(cfg)

evaluated_prices = EvaluatedPrices(prices, cx, overrides=overrides)

labor = EvaluatedLabor(evaluated_prices)

buildings = prundf.PrunBuildings()

analysis = BuildingAnalysis(cx=cx, buildings=prundf.PrunBuildings(), prices=evaluated_prices, materials=prundf.PrunMaterials(), labor=labor)

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(-1)
pl.Config.set_thousands_separator(",")
pl.Config.set_tbl_rows(-1)

polars.config.Config

In [8]:
a_df = analysis.df(1.40)
a_df = (a_df.join(buildings.source_df, left_on=("Building"), right_on="Ticker")
        .drop("Area","Name"))
avg_volume_df = (cxpc_all.avg_volume_df
                 .group_by("Ticker")
                 .agg(pl.col("7DayAvgVolume").get(pl.col("ts").arg_max()).alias("Volume"),
                      pl.col("ts").max().alias("latest_ts")))
a_df = (a_df.join(avg_volume_df,left_on="OutputMaterial", right_on="Ticker"))
        
#print(analysis.unit_efficiency_df.filter(pl.col("Recipe").str.contains("SF").and_(pl.col("Building") == "REF")))
print(a_df
        .drop("RevenuePerRun","TotalLaborCostPerDay","InputCostPerRun")
        .filter(pl.col("Expertise")=="MANUFACTURING")
        .filter(pl.col("Building")!="SPF")
        .filter(pl.col("Volume")>750000)
        .filter(pl.col("TotalCostPerDay")<60000)
        .sort("ProfitPerDay", descending=True)
        .head(20))

# TODO: join volume and filter out low volume

shape: (10, 15)
┌──────────┬───────────────┬─────────────────────────────────┬────────────┬─────────────┬────────────────┬────────────────┬───────────────┬─────────────────┬─────────────────┬──────────────┬───────────────┬───────────────┬──────────────────┬─────────────────────┐
│ Building ┆ Duration      ┆ Recipe                          ┆ RunsPerDay ┆ UnitsPerDay ┆ OutputMaterial ┆ OutputQuantity ┆ RevenuePerDay ┆ InputCostPerDay ┆ TotalCostPerDay ┆ CostPerUnit  ┆ ProfitPerDay  ┆ Expertise     ┆ Volume           ┆ latest_ts           │
│ ---      ┆ ---           ┆ ---                             ┆ ---        ┆ ---         ┆ ---            ┆ ---            ┆ ---           ┆ ---             ┆ ---             ┆ ---          ┆ ---           ┆ ---           ┆ ---              ┆ ---                 │
│ str      ┆ f64           ┆ str                             ┆ f64        ┆ f64         ┆ str            ┆ i64            ┆ f64           ┆ f64             ┆ f64             ┆ f64          ┆ f

In [9]:
print(analysis.df(efficiency=1.25).filter(pl.col("Recipe").str.contains("SF").and_(pl.col("Building") == "REF")))

shape: (2, 15)
┌──────────┬──────────┬─────────────────┬─────────────────────────┬────────────┬─────────────┬────────────────┬───────────────┬────────────────┬───────────────┬─────────────────┬──────────────────────┬─────────────────┬─────────────┬──────────────┐
│ Building ┆ Duration ┆ InputCostPerRun ┆ Recipe                  ┆ RunsPerDay ┆ UnitsPerDay ┆ OutputMaterial ┆ RevenuePerRun ┆ OutputQuantity ┆ RevenuePerDay ┆ InputCostPerDay ┆ TotalLaborCostPerDay ┆ TotalCostPerDay ┆ CostPerUnit ┆ ProfitPerDay │
│ ---      ┆ ---      ┆ ---             ┆ ---                     ┆ ---        ┆ ---         ┆ ---            ┆ ---           ┆ ---            ┆ ---           ┆ ---             ┆ ---                  ┆ ---             ┆ ---         ┆ ---          │
│ str      ┆ f64      ┆ f64             ┆ str                     ┆ f64        ┆ f64         ┆ str            ┆ f64           ┆ i64            ┆ f64           ┆ f64             ┆ f64                  ┆ f64             ┆ f64         ┆ f64 